<a href="https://colab.research.google.com/github/gabrielmprata/anatel_banda_larga_fixa/blob/main/PREP_banda_larga_fixa_pwr_bi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img loading="lazy" src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/python/python-original.svg" width="40" height="40"/> <img src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/pandas/pandas-original-wordmark.svg" width="40" height="40"/>   <img loading="lazy" src="https://img.icons8.com/?size=100&id=3sGOUDo9nJ4k&format=png&color=000000" width="40" height="40"/>

---

>
**Dev**: Gabriel Prata
>
**Data**: 13/05/2026
>
**Última modificação**: 13/05/2026
>
**Contexto**: *Preparar os Dados abertos de banda larga fixa.*
>
---

Pré-processamento dos dados, para serem utilizados em um painel elaborado em Power BI.
>
<img loading="lazy" src="https://img.icons8.com/?size=100&id=3sGOUDo9nJ4k&format=png&color=000000" width="40" height="40"/>

>
![Badge em Desenvolvimento](http://img.shields.io/static/v1?label=STATUS&message=EM%20DESENVOLVIMENTO&color=GREEN&style=for-the-badge)

![Badge versao](http://img.shields.io/static/v1?label=Ver.&message=v3.0&color=red&style=for-the-badge&logo=github)

#**<font color=#85d338 size="6"> Import libraries**

In [1]:
# Importação de pacotes
import pandas as pd
import numpy as np
import missingno as ms # para tratamento de missings
import datetime
import calendar
import re # expressão regulares
import unicodedata

#bibliotecas para visualização de dados
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#compactar
from shutil import copyfileobj
import bz2
# Data e hora
from datetime import datetime, date, time

# Configuração para não exibir os warnings
import warnings
warnings.filterwarnings("ignore")

#**<font color=#85d338 size="6"> 1. Coleta de dados**

As informações de acessos de **Banda Larga Fixa**, estão no sítio de dados abertos do governo federal, no link abaixo:
>
https://dados.gov.br/dados/conjuntos-dados/acessos---banda-larga-fixa

>
É disponibilizado um arquivo ZIP com todos os anos disponíveis.
>
Para esse projeto o arquivo foi coletado no dia **13/05/2026**.
>
Selecionamos os seguintes arquivos de **2025**, para tratar:
>
```
Acessos_Banda_Larga_Fixa_2025.zip
```
>
E o arquivo de **2024**, para analisar a evolução:
>
```
Acessos_Banda_Larga_Fixa_2024.zip
```

In [2]:
# importando dataset

# Conjunto de dados com os acessos de 2025
df_acessos_bl_2025 = pd.read_csv("Acessos_Banda_Larga_Fixa_2025.csv.bz2", compression='bz2', delimiter=';')

# Conjunto de dados com os acessos de 2024
df_acessos_bl_2024 = pd.read_csv("Acessos_Banda_Larga_Fixa_2024.csv.bz2", compression='bz2', delimiter=';')


No dataset de 2024, só temos interesse no mês de **DEZEMBRO**.
>
Vamos criar um Dataframe selecionado, e depois concatenar com o Dataframe de 2025.

In [3]:
df_acessos_bl_2024 = df_acessos_bl_2024[(df_acessos_bl_2024['Mês'] == 12)].copy()

Utilizei a função "concat" para juntar os DataFranes, em apenas um.

In [4]:
df_acessos_bl = pd.concat([df_acessos_bl_2025, df_acessos_bl_2024],sort=False, ignore_index=True)



---



#**<font color=#85d338 size="6"> 2. Análise de Dados Inicial**

###**<font color=#85d338> 2.1. Estatísticas Descritivas**

Compreende a organização, o resumo e, descrever os dados, que podem ser expressos em tabelas e gráficos.
>
Veremos a seguir alguns comandos para exibir algumas estatísticas descritivas.


In [ ]:
#	Quantidade de atributos e instâncias (linhas/colunas)
df_acessos_bl.shape

<font color=#85d338> Data frame com 16 atributos(features) e cerca de 8.8 milhões de tuplas.
>


---





In [ ]:
# Exibir os 5 primeiros registros
df_acessos_bl.head(5)



---



In [ ]:
# Mostra diversas informações do Dataframe em um único comando, e exibir o uso de memória
df_acessos_bl.info(memory_usage="deep")

<font color=#85d338> Data frame com cerca de **6 gigas de memória**.


---

In [ ]:
# Quantidade de valores únicos
df_acessos_bl.nunique()



---



In [ ]:
# Quantidade de NaN/Missing/Nulls no dataframe
df_acessos_bl.isnull().sum()



---



###**<font color=#85d338> 2.2. Distribuição dos atributos**

>Nessa etapa, iremos verificar a distribuição dos principais atributos. Para ver se existe a necessidade de tomar alguma ação de transformações na etapa de preparação de dados.


---

In [ ]:
df_acessos_bl.describe().round(2)



---



#**<font color=#85d338 size="6"> 3. Pré-Processamento de dados**

Após coletar e analisar os dados na etapa anterior, agora é o momento
de limpar, transformar e apresentar melhor os dados.
>
Assim poderemos obter, na próxima etapa, os melhores resultados possíveis nos algoritmos de
Machine Learning, ou simplesmente apresentar dados mais confiáveis para os clientes em soluções de
business intelligence.


---

###**<font color=#85d338> 3.1. Limpeza**

De forma resumida, a limpeza consiste na verificação da consistência das informações, correção de possíveis erros de preenchimento ou eliminação de valores desconhecidos, redundantes ou não pertencentes ao domínio.



####**<font color=#85d338> 3.1.1 Padronização de dados**

Dentro da programação, possuímos alguns padrões de escrita para nomes de variáveis, funções, classes e assim por diante.
>
Esses padrões de escrita são chamados de estilos de case.
>
Existem diversos tipos de case, nesse projeto iremos utilizar:
>
**Snake Case (snake_case)**: Nesse estilo, todas as letras são minúsculas e as palavras são separadas por um underscore(_).
>
**Remover acentos (remover_acentos)**: Nesse estilo, todas as letras são minúsculas e as palavras são separadas por um underscore(_).

In [ ]:
def remover_acentos(texto):
    # Decompõe caracteres acentuados (ex: á -> a + ´)
    texto_normalizado = unicodedata.normalize('NFKD', texto)
    # Remove os caracteres de acentuação (intervalo unicode)
    return re.sub(r'[\u0300-\u036f]', '', texto_normalizado)

df_acessos_bl.columns = [remover_acentos(column) for column in df_acessos_bl.columns]
df_acessos_bl.columns

Aplicar **Snake Case**

In [ ]:
# Criar uma função para aplicar o snake_case
def snake_case(string):
    string = re.sub(" +", " ", string)   # substitui múltiplos espaços por um espaço
    string = re.sub(" ", "_", string)    # substitui espaço por _
    return string.lower() # transforma em minuscula

df_acessos_bl.columns = [snake_case(column) for column in df_acessos_bl.columns]
df_acessos_bl.columns

####**<font color=#85d338> 3.1.2 Redundâncias**

Vamos eliminar as colunas que não iremos utilizar em nossas analises.
>
A conta de estudante no Power BI, suporta apenas arquivos de 1Gb, sendo assim, terei que excluir as informações de município e depois agregar para reduzir a quantidade de linhas.
>
A ideia é ter um dataframe mais leve, e com pouco espaço em disco.

In [ ]:
df_acessos_bl.drop([
				            'cnpj','tecnologia'
                    ,'tipo_de_pessoa'
                    ,'tipo_de_produto'
                    ,'municipio'
                    ,'codigo_ibge_municipio'
                    ], axis=1, inplace= True)

In [ ]:
df_acessos_bl.info()

####**<font color=#85d338> 3.1.3 Tratamento de Missings**

Como o DataFrame tem muitos atributos, e muitos deles possuem valores nulos, vou utlizar um método para mostrar apenas os valores nulos e o percentual.
>
1) Converter a função isnull() em um pd.series e depois transformar em DF, assim conseguimos filtrar quais atributos são nulos.
>

In [ ]:
#1)
# cria um pd.series
dfnull = df_acessos_bl.isnull().sum()

# Converte series em dataframe
dfnull = (dfnull.to_frame(name="QTD"))

tot = len(df_acessos_bl) #Total de registros no dataset

#Criano o atributo perc, para saber o percentual de registros nulo do atributo
dfnull["perc"] = ((dfnull['QTD']/tot)*100).round(2)

#Mostrar apenas os atributos com valores nulos, ordenando para o com mais linhas nulas
dfnull.query('QTD > 0').sort_values(by='perc', ascending=False)

In [ ]:
df_acessos_bl['grupo_economico'] = df_acessos_bl['grupo_economico'].fillna('OUTROS')
df_acessos_bl['empresa'] = df_acessos_bl['empresa'].fillna('Não identificado')
df_acessos_bl['porte_da_prestadora'] = df_acessos_bl['porte_da_prestadora'].fillna('Pequeno Porte')




---



###**<font color=#85d338> 3.2 Criação de recursos**

><font color=#85d338>**Atributo data completa**

Criar um atributo do tipo DATE TIME para o Power BI criar a hierarquia de datas.

In [ ]:
df_acessos_bl['ano_mes'] = df_acessos_bl['ano'].map(str) + '-' + df_acessos_bl['mes'].map(str)
df_acessos_bl['ano_mes'] = pd.to_datetime(df_acessos_bl['ano_mes'])
df_acessos_bl['ano_mes'] = df_acessos_bl['ano_mes'].dt.strftime('%Y-%m')

In [ ]:
df_acessos_bl['data'] = pd.to_datetime(df_acessos_bl['ano_mes'])

In [ ]:
# excluir atributo
df_acessos_bl.drop(['ano_mes'], axis=1, inplace= True)

###**<font color=#85d338> 3.3 Redução da dimensionalidade**

Os datasets podem ter muitas características, e muitos algoritmos de Machine Learning funcionam melhor se a dimensionalidade for menor.

####**<font color=#85d338> 3.3.1 Agregação**

Também pode ser considerada uma técnica de redução de dimensionalidade, pois reduz o número de colunas e linhas do dataset.
>
O nosso dataset, é aberto por Municipio, e nesse momento não iremos analisar nessa granularidade, pois a conta gratuita do Power BI, permite arquivos de até 1Gb.
>
Vamos criar um dataset agregado com a visão por UF, agrupandos os atributos que não se repetem.
>
Assim, será possivel fazer analises mais rápidas, e ter um dataset menor para usar no Github e no Power BI.

In [ ]:
df_acessos_bl.info()

In [ ]:
# Agregando as informações
df_acessos_bl_fixa = df_acessos_bl.groupby(["ano",
                                              "mes",
                                              "grupo_economico",
                                              "empresa",
                                              "porte_da_prestadora",
                                              "uf",
                                              "meio_de_acesso",
                                            "faixa_de_velocidade",
                                            "velocidade"])['acessos'].sum().reset_index()

In [ ]:
df_acessos_bl_fixa.info()


Uma redução de 8.8 milhões de registros para 1.6M .
>
Uso de memória de 740Mb para 128Mb.

Na próxima etapa, iremos melhorar a faixa de velocidade, e depois iremos descartar esses atributos, reduzindo novamente a dimensionalidade do dataset.

###**<font color=#85d338> 3.4 Feature engineering**

Consiste em criar, a partir dos atributos originais, um conjunto de atributos que capture informações importantes.
>
Iremos criar outros datasets reduzidos para facilitar a leitura dos dados.

####**<font color=#85d338> 3.4.1 Faixa de velocidade**

In [ ]:
df_acessos_bl_fixa.groupby('faixa_de_velocidade').size().sort_values(ascending=False)

In [ ]:
# COnverter o atributo para float
df_acessos_bl_fixa['velocidade'] = (df_acessos_bl_fixa['velocidade'].str.replace(',','.')).astype('float')

In [ ]:
df_acessos_bl_fixa.groupby('velocidade').size().sort_values(ascending=False)

In [ ]:
classes = [0, 33, 50, 100, 200, 300,400,500, 600, 700, 800, 10000000]

labels = ['< 34Mbps', 'Entre 34 e 50Mbps', 'Entre 50 e 100Mbps', 'Entre 100 e 200Mbps',
          'Entre 200 e 300Mbps','Entre 300 e 400Mbps','Entre 400 e 500Mbps','Entre 500 e 600Mbps','Entre 600 e 700Mbps','Entre 700 e 800Mbps','800 +']

classes = pd.cut(x=df_acessos_bl_fixa.velocidade, bins=classes, labels=labels)
df_acessos_bl_fixa['faixa_velocidade'] = classes

In [ ]:
df_acessos_bl_fixa.groupby('faixa_velocidade').size().sort_values(ascending=False)

**Pronto, agora iremos excluir o atributo velocidade e a faixa de velocidade antiga, e agregar novamente o dataset.**

In [ ]:
df_acessos_bl_fixa.drop([
				            'faixa_de_velocidade'
                    ,'velocidade'], axis=1, inplace= True)

In [ ]:
# Agregando as informações
df_acessos_bl_fixa = df_acessos_bl_fixa.groupby(["ano",
                                              "mes",
                                              "grupo_economico",
                                              "empresa",
                                              "porte_da_prestadora",
                                              "uf",
                                              "meio_de_acesso",
                                              "faixa_velocidade"])['acessos'].sum().reset_index()

In [ ]:
df_acessos_bl_fixa.info()

In [ ]:
df_acessos_bl_fixa.info()

###**<font color=#85d338> 3.4 Export**

Agora iremos exportar o dataset em CSV, para usarmos no Power BI

In [ ]:
# Exportar para csv
df_anuario.to_csv('df_anuario.csv', sep='|', encoding="Latin 1", index=False)

df_cisp.to_csv('df_cisp.csv', sep='|', encoding="Latin 1", index=False)

df_hist_anual.to_csv('df_hist_anual.csv', index=False)
df_hs_compara.to_csv('df_hs_compara.csv', sep=';', index=False)
df_regiao_cisp.to_csv('df_regiao_cisp.csv', sep='|', index=False)
df_municipio_taxa.to_csv('df_municipio_taxa.csv', index=False)
df_hs_taxa.to_csv('df_hs_taxa.csv', index=False)
df_frota.to_csv('df_frota.csv', index=False)
df_hs_armas.to_csv('df_hs_armas.csv', sep=';', index=False)

# compactar arquivo com nivel de compressão máxima
with open('df_anuario.csv', 'rb') as input:
    with bz2.BZ2File('df_anuario.csv.bz2', 'wb', compresslevel=9) as output:
        copyfileobj(input, output)

In [ ]:
df_acessos_bl.to_csv('df_acessos_bl.csv', sep=';', index=False)

In [ ]:
df_acessos_bl.ano.value_counts(normalize=True)

In [ ]:
df_acessos_bl.info()

In [ ]:
df_acessos_bl.drop(df_acessos_bl[df_acessos_bl['ano'] == 2024].index, inplace=True)